# Compare streaming conditions

This notebook aggregates multiple runs from the streaming experiment (clean vs poisoned conditions) and overlays key metrics across seeds.

In [ ]:
from __future__ import annotations

from pathlib import Path
import json

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

sns.set(style="whitegrid")

ROOT = Path("..").resolve()
OUT = ROOT / "outputs"

print(f"ROOT: {ROOT}")
print(f"OUT: {OUT}")


def _iter_condition_seed_dirs(conditions, base_dir: Path = OUT):
    """Yield (condition, seed, run_dir) for existing runs only.

    Skips missing folders gracefully.
    """
    for cond in conditions:
        cond_dir = base_dir / str(cond)
        if not cond_dir.exists():
            continue
        for sub in sorted(cond_dir.glob("seed_*")):
            if not sub.is_dir():
                continue
            try:
                seed = int(sub.name.split("_")[-1])
            except Exception:
                continue
            yield cond, seed, sub


def load_runs(conditions, base_dir: Path = OUT):
    """Load metrics and TDA features for all available runs.

    Returns
    -------
    metrics_long : DataFrame
        Columns: condition, seed, t, test_acc.
    tda_long : DataFrame
        Columns: condition, seed, t, score, h1_max_persistence.
    configs : dict
        Mapping (condition, seed) -> loaded config dict.
    """
    metrics_rows = []
    tda_rows = []
    configs: dict[tuple[str, int], dict] = {}

    for cond, seed, run_dir in _iter_condition_seed_dirs(conditions, base_dir):
        metrics_path = run_dir / "metrics.csv"
        tda_path = run_dir / "tda_features.csv"
        config_path = run_dir / "config.json"

        if not metrics_path.exists() or not tda_path.exists():
            continue

        # Config (optional but useful for threshold/poison window).
        if config_path.exists():
            with config_path.open("r", encoding="utf-8") as f:
                configs[(cond, seed)] = json.load(f)

        m = pd.read_csv(metrics_path)
        if "t" not in m.columns or "test_accuracy" not in m.columns:
            continue
        m_long = m[["t", "test_accuracy"]].copy()
        m_long["condition"] = cond
        m_long["seed"] = seed
        m_long = m_long.rename(columns={"test_accuracy": "test_acc"})
        metrics_rows.append(m_long)

        tdf = pd.read_csv(tda_path)
        if "t" not in tdf.columns:
            continue
        # score and h1_max_persistence are optional; fall back to NaN.
        cols = {"t": tdf["t"]}
        cols["score"] = tdf["score"] if "score" in tdf.columns else np.nan
        cols["h1_max_persistence"] = (
            tdf["h1_max_persistence"] if "h1_max_persistence" in tdf.columns else np.nan
        )
        t_long = pd.DataFrame(cols)
        t_long["condition"] = cond
        t_long["seed"] = seed
        tda_rows.append(t_long)

    metrics_long = pd.concat(metrics_rows, ignore_index=True) if metrics_rows else pd.DataFrame(
        columns=["condition", "seed", "t", "test_acc"]
    )
    tda_long = pd.concat(tda_rows, ignore_index=True) if tda_rows else pd.DataFrame(
        columns=["condition", "seed", "t", "score", "h1_max_persistence"]
    )

    return metrics_long, tda_long, configs


def get_label_flip_config(configs: dict[tuple[str, int], dict]):
    """Return (config, seed) for label_flip/seed_0 if present, else first label_flip config.

    Falls back to (None, None) if no label_flip config is available.
    """
    key = ("label_flip", 0)
    if key in configs:
        return configs[key], 0
    # Fallback: any label_flip run.
    for (cond, seed), cfg in configs.items():
        if cond == "label_flip":
            return cfg, seed
    return None, None


def get_poison_window_and_threshold(configs: dict[tuple[str, int], dict]):
    """Extract poison window and threshold from label_flip config if available."""
    cfg, seed = get_label_flip_config(configs)
    if cfg is None:
        return None, None, np.nan

    poison_start = cfg.get("poison_start_t")
    poison_end = cfg.get("poison_end_t")
    threshold = cfg.get("threshold", np.nan)
    print(f"Using label_flip config from seed={seed}: start={poison_start}, end={poison_end}, threshold={threshold}")
    return poison_start, poison_end, threshold


conditions = ["clean", "label_flip", "trigger", "drift", "drift_poison"]
metrics_long, tda_long, configs = load_runs(conditions)

print("Loaded metrics_long shape:", metrics_long.shape)
print("Loaded tda_long shape:", tda_long.shape)

metrics_long.head(), tda_long.head()

In [ ]:
# Plot test accuracy vs t (clean vs label_flip)
if not metrics_long.empty:
    subset = metrics_long[metrics_long["condition"].isin(["clean", "label_flip"])]
    if not subset.empty:
        plt.figure(figsize=(8, 4))
        sns.lineplot(
            data=subset,
            x="t",
            y="test_acc",
            hue="condition",
            errorbar="sd",
        )
        poison_start, poison_end, _ = get_poison_window_and_threshold(configs)
        ax = plt.gca()
        if poison_start is not None and poison_end is not None:
            ax.axvspan(poison_start, poison_end, color="red", alpha=0.1, label="poison window")
            handles, labels = ax.get_legend_handles_labels()
            # Ensure poison window label appears once.
            unique = dict(zip(labels, handles))
            ax.legend(unique.values(), unique.keys())
        ax.set_title("Test accuracy over time (mean ± sd across seeds)")
        plt.tight_layout()
        plt.show()
    else:
        print("No metrics available for clean/label_flip.")
else:
    print("metrics_long is empty; run experiments first.")

In [ ]:
# Plot detection score vs t (clean vs label_flip)
if not tda_long.empty:
    subset = tda_long[tda_long["condition"].isin(["clean", "label_flip"])]
    if not subset.empty and subset["score"].notna().any():
        plt.figure(figsize=(8, 4))
        sns.lineplot(
            data=subset,
            x="t",
            y="score",
            hue="condition",
            errorbar="sd",
        )
        poison_start, poison_end, threshold = get_poison_window_and_threshold(configs)
        ax = plt.gca()
        if not np.isnan(threshold):
            ax.axhline(threshold, color="orange", linestyle="--", label="threshold")
        if poison_start is not None and poison_end is not None:
            ax.axvspan(poison_start, poison_end, color="red", alpha=0.1, label="poison window")
        handles, labels = ax.get_legend_handles_labels()
        unique = dict(zip(labels, handles))
        ax.legend(unique.values(), unique.keys())
        ax.set_title("Detection score over time (mean ± sd across seeds)")
        plt.tight_layout()
        plt.show()
    else:
        print("No score data available for clean/label_flip.")
else:
    print("tda_long is empty; run experiments first.")

In [ ]:
# Plot H1 max persistence vs t (clean vs label_flip)
if not tda_long.empty and "h1_max_persistence" in tda_long.columns:
    subset = tda_long[tda_long["condition"].isin(["clean", "label_flip"])]
    if not subset.empty and subset["h1_max_persistence"].notna().any():
        plt.figure(figsize=(8, 4))
        sns.lineplot(
            data=subset,
            x="t",
            y="h1_max_persistence",
            hue="condition",
            errorbar="sd",
        )
        poison_start, poison_end, _ = get_poison_window_and_threshold(configs)
        ax = plt.gca()
        if poison_start is not None and poison_end is not None:
            ax.axvspan(poison_start, poison_end, color="red", alpha=0.1, label="poison window")
            handles, labels = ax.get_legend_handles_labels()
            unique = dict(zip(labels, handles))
            ax.legend(unique.values(), unique.keys())
        ax.set_title("H1 max persistence over time (mean ± sd across seeds)")
        plt.tight_layout()
        plt.show()
    else:
        print("No H1 persistence data available for clean/label_flip.")
else:
    print("tda_long is empty or missing h1_max_persistence; run experiments first.")